In [0]:
# ==============================================================================
# MEDALLION PIPELINE: TRANSFORMING BRONZE TO SILVER (PYSPARK)
# USE CASES: TALENT ASSET & RISK AUDIT MANAGEMENT
# ==============================================================================

from pyspark.sql import functions as F
from pyspark.sql.window import Window

# --- SETUP SILVER SCHEMA ---
spark.sql("CREATE SCHEMA IF NOT EXISTS training_medallion.02_silver")

# 1. READ RAW ASSETS FROM BRONZE LAYER
df_events_bronze = spark.table("training_medallion.01_bronze.bronze_talent_risk_events")
df_params_bronze = spark.table("training_medallion.01_bronze.bronze_department_risk_parameters")

# ==============================================================================
# SECTION I: DATA CLEANSING (BOTTOM BRANCHES OF MIND MAP)
# ==============================================================================

# A. Data Filtering: Filter out invalid rows with null audit_id
df_filtered = df_events_bronze.filter(F.col("audit_id").isNotNull())

# B. Handling Unwanted Spaces: Strip extra whitespaces
df_trimmed = (
    df_filtered
    .withColumn("flag_reason", F.trim(F.regexp_replace(F.col("flag_reason"), " +", " ")))
)

# C. Data Casting & Timestamp Normalization: Cast types and parse timestamps
df_casted = (
    df_trimmed
    .withColumn("employee_id", F.col("employee_id").cast("integer"))
    .withColumn("risk_score", F.col("risk_score").cast("double"))
    .withColumn("event_timestamp", F.to_timestamp(F.col("ingestion_timestamp"), "yyyy-MM-dd HH:mm:ss"))
    .withColumn("event_date", F.to_date(F.col("event_timestamp")))
)

# D. Handling Missing Data (Null Handling) & Invalid Values
df_imputed = (
    df_casted
    # Replace negative or null risk scores with default fallback (0.0)
    .withColumn(
        "risk_score",
        F.when((F.col("risk_score").isNull()) | (F.col("risk_score") < 0), F.lit(0.0))
         .otherwise(F.col("risk_score"))
    )
    .withColumn(
        "flag_reason",
        F.when(F.col("flag_reason").isNull(), F.lit("Standard Check"))
         .otherwise(F.col("flag_reason"))
    )
)

# E. Outlier Detection: Identify and cap extreme risk values (> 100)
df_outliers = (
    df_imputed
    .withColumn("is_outlier", F.when(F.col("risk_score") > 100, F.lit(True)).otherwise(F.lit(False)))
    .withColumn("risk_score_capped", F.when(F.col("risk_score") > 100, F.lit(100.0)).otherwise(F.col("risk_score")))
)

# F. Remove Duplicates: Deduplicate records based on the most recent event timestamp
window_dedup = Window.partitionBy("audit_id").orderBy(F.col("event_timestamp").desc())

df_cleansed = (
    df_outliers
    .withColumn("row_num", F.row_number().over(window_dedup))
    .filter(F.col("row_num") == 1)
    .drop("row_num")
)

# ==============================================================================
# SECTION II: TRANSFORMATION & ENRICHMENT (TOP BRANCHES OF MIND MAP)
# ==============================================================================

# A. Naming Convention: Standardize column names into snake_case
df_renamed = (
    df_cleansed
    .withColumnRenamed("flag_reason", "audit_reason_detail")
    .withColumnRenamed("risk_score_capped", "standardized_risk_score")
)

# B. Data Integration & Data Enrichment: Join with department risk parameters
df_enriched = (
    df_renamed.join(
        df_params_bronze,
        df_renamed.standardized_risk_score <= df_params_bronze.max_risk_threshold,
        how="left"
    )
    .select(
        df_renamed["*"],
        df_params_bronze["department_name"],
        df_params_bronze["compliance_level"]
    )
    .na.fill({"department_name": "General Operations", "compliance_level": "Standard"})
)

# C. Business Rules & Logic: Classify talent evaluation action category
df_business_rules = (
    df_enriched
    .withColumn(
        "talent_action_status",
        F.when(F.col("standardized_risk_score") >= 80, F.lit("Immediate Audit Review"))
         .when(F.col("standardized_risk_score") >= 50, F.lit("Requires Additional Training"))
         .otherwise(F.lit("Fully Compliant"))
    )
)

# D. Derived Column: Flag automated compliance escalation
df_derived = (
    df_business_rules
    .withColumn(
        "is_escalation_required",
        F.when((F.col("talent_action_status") == "Immediate Audit Review") & (F.col("compliance_level") == "Strict"), F.lit(True))
         .otherwise(F.lit(False))
    )
    .withColumn("silver_processed_at", F.current_timestamp())
)

# E. Data Aggregations: Compute aggregated employee summary metrics
df_aggregated_summary = (
    df_derived
    .groupBy("employee_id")
    .agg(
        F.count("audit_id").alias("total_audit_events"),
        F.avg("standardized_risk_score").alias("avg_employee_risk_score"),
        F.max("standardized_risk_score").alias("peak_risk_score"),
        F.sum(F.when(F.col("is_escalation_required") == True, 1).otherwise(0)).alias("escalation_count")
    )
)

# ==============================================================================
# SECTION III: WRITE TO SILVER DELTA TABLES
# ==============================================================================

# 1. Cleaned Events Table
(
    df_derived.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("training_medallion.02_silver.silver_talent_risk_cleansed")
)

# 2. Aggregated Employee Profile Table
(
    df_aggregated_summary.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("training_medallion.02_silver.silver_employee_risk_profile")
)

print("Bronze-to-Silver transformation completed successfully.")

Bronze-to-Silver transformation completed successfully.
